# Setup do ambiente de execução (estágio 00)

Prepara o ambiente de forma idêntica no Google Colab e no Kaggle: entrega do pacote `src/`, instalação das versões pinadas, acesso ao armazenamento canônico em `MyDrive/tcc/`, autenticação do Earth Engine, fixação de sementes e flags determinísticas e validação do setup.

## Bootstrap do workspace

O primeiro passo baixa e executa `src/bootstrap.py` (somente stdlib) — necessário porque o `src/` ainda não está disponível para import. O bootstrap obtém o repositório público, extrai `src/`, `data/external/` e `requirements-runtime.txt` para o workspace, adiciona o workspace ao `sys.path` e, no Colab, cria o espelho `MyDrive/tcc/repo/`. O `reload` garante que reexecuções usem a versão mais recente baixada.

In [ ]:
# Baixa e executa o bootstrap do workspace (etapa prévia ao import de src/).
import importlib
import pathlib
import sys
import urllib.request

BOOTSTRAP_URL = "https://raw.githubusercontent.com/jotap1101/tcc/main/src/bootstrap.py"
pathlib.Path("bootstrap.py").write_bytes(urllib.request.urlopen(BOOTSTRAP_URL).read())
sys.path.insert(0, str(pathlib.Path.cwd()))

# Recarrega o módulo para não reutilizar uma versão antiga em cache no kernel.
bootstrap = importlib.import_module("bootstrap")
importlib.reload(bootstrap)

workspace = bootstrap.bootstrap_workspace()
print(f"Workspace: {workspace}")

## Dependências pinadas

Instala as versões fixadas em `requirements-runtime.txt`, garantindo o mesmo conjunto de bibliotecas nas duas plataformas.

In [ ]:
# Instala as versões pinadas do requirements-runtime.txt no ambiente atual.
import subprocess
import sys

requirements = pathlib.Path(workspace) / "requirements-runtime.txt"
subprocess.run(
    [sys.executable, "-m", "pip", "install", "-r", str(requirements)],
    check=True,
)
print("Dependências instaladas a partir de:", requirements)

## Contexto de runtime (plataforma e armazenamento)

Importa o pacote `src/` (já entregue pelo bootstrap) e resolve o contexto de runtime: plataforma de execução, raiz canônica `MyDrive/tcc/` com todas as subpastas definidas em `src/config.yaml` (reutilizadas quando existem; criadas quando não — no Kaggle via Drive API com cache local) e a configuração carregada.

In [ ]:
# Resolve o contexto de runtime: plataforma, armazenamento canônico e configuração.
from src import io
from src.setup import initialize

ctx = initialize(workspace)
platform = ctx.platform
storage_paths = ctx.storage_paths
config = ctx.config
print(f"Plataforma: {platform}")
for key, path in storage_paths.items():
    print(f"{key}: {path}")

## Autenticação do Earth Engine

Autentica o GEE com a conta principal: fluxo interativo no Colab; no Kaggle, carrega a credencial da variável `GEE_CREDENTIALS`. A lógica de plataforma fica isolada em `src/`.

In [ ]:
# Autentica e inicializa o Earth Engine com a conta principal.
from src.utils import authenticate_gee

authenticate_gee()
print("Earth Engine autenticado e inicializado.")

## Reprodutibilidade

Fixa sementes (python/numpy/torch/cuda), habilita flags determinísticas do PyTorch e registra as versões das bibliotecas instaladas.

In [ ]:
# Fixa sementes, flags determinísticas e registra as versões do ambiente.
from src.utils import log_environment, setup_reproducibility

setup_reproducibility(config)
log_environment()

## Validação do setup

Verifica se a malha IBGE versionada está acessível e se a configuração carregou corretamente.

In [ ]:
# Valida o setup: malha IBGE acessível e configuração carregada.
mesh_path = workspace / config["aoi"]["mesh_path"]
print(f"Malha IBGE acessível: {mesh_path.is_file()}")
print(f"Configuração carregada: {len(config)} blocos no nível raiz")
print("Setup concluído.")